# Scout de Jóvenes Promesas — Paso 3
## Simulación Montecarlo — Proyección de Valor Futuro

**Input:**
- `outputs/paso2/scouting_dataset_v2_con_predicciones.csv` (o v1 si no está disponible)
- `data/player_market_value.csv` → para calibrar las tasas de crecimiento históricas

**Objetivo de este paso:**
El modelo del Paso 2 predice el valor de mercado *actual* de un jugador.
Pero un reclutador necesita saber: **¿cuánto puede valer este jugador en 3 o 5 años?**

La simulación Montecarlo responde esa pregunta generando **10,000 trayectorias
posibles** para cada jugador, basadas en cómo han evolucionado históricamente
los valores de jugadores con perfiles similares (misma posición, misma banda de edad).
El resultado es una **distribución de probabilidad** del valor futuro, no un
número único — porque el futuro de un jugador tiene incertidumbre real.

**Por qué Montecarlo y no una proyección lineal simple:**
Una proyección lineal asumiría que el jugador siempre crece al mismo ritmo.
Montecarlo captura que el crecimiento es estocástico: a veces el mercado
explota el valor de un jugador joven, a veces una lesión lo derrumba,
a veces se estanca. La distribución resultante refleja esa incertidumbre real.

In [2]:
# Imports y configuración
# ===========================================================

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from pathlib import Path

warnings.filterwarnings("ignore")

DATA_V1    = Path("../outputs/paso1/scouting_dataset_v1.csv")
DATA_V2    = Path("../outputs/paso2/scouting_dataset_v2_con_predicciones.csv")
MV_HIST    = Path("../data/player_market_value.csv")
PROFILES   = Path("../data/player_profiles.csv")
OUTPUT_DIR = Path("../outputs/paso3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SIMULATIONS = 10_000   # simulaciones por jugador
HORIZONS      = [1, 3, 5] # años a proyectar
RANDOM_SEED   = 42

PALETTE = {
    "primary":   "#1D9E75",
    "secondary": "#7F77DD",
    "accent":    "#EF9F27",
    "danger":    "#D85A30",
    "neutral":   "#888780",
    "light":     "#F4F3EE",
    "dark":      "#2C2C2A",
}
plt.rcParams.update({
    "figure.facecolor": PALETTE["light"],
    "axes.facecolor":   PALETTE["light"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size": 11,
})

def save_fig(fig, name):
    path = OUTPUT_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=PALETTE["light"])
    plt.close(fig)
    print(f"  ✓ {name}.png")

np.random.seed(RANDOM_SEED)
print("✓ Configuración lista")
print(f"  Simulaciones por jugador: {N_SIMULATIONS:,}")
print(f"  Horizontes de proyección: {HORIZONS} años")

✓ Configuración lista
  Simulaciones por jugador: 10,000
  Horizontes de proyección: [1, 3, 5] años


## 1. Calibración de la distribución de crecimiento

El primer paso es aprender del historial: **¿a qué tasa crece el valor
de un jugador de 20 años delantero? ¿y uno de 25 años defensor?**

Se calculan las tasas de crecimiento anuales reales para 394,243 transiciones
de valor del dataset histórico (2003–2025), agrupadas por posición y banda de edad.

**¿Por qué log-normal?**

Las tasas de crecimiento son siempre positivas y tienen cola derecha larga
(algunos jugadores se disparan x10 en un año, la mayoría crece moderadamente).
Cuando se aplica logaritmo a las tasas, la distribución resultante es
aproximadamente normal — esa es la definición de log-normal.

Se parametriza con dos valores por combinación posición × banda:
- **μ (mu):** media del logaritmo de la tasa → controla el crecimiento esperado
- **σ (sigma):** desviación estándar del logaritmo → controla la incertidumbre

Un μ positivo indica crecimiento esperado, negativo indica declive esperado.

In [3]:
# Carga de datos
# ===========================================================

# Dataset principal — usar v2 si existe (tiene predicciones del modelo)
if DATA_V2.exists():
    df = pd.read_csv(DATA_V2, low_memory=False)
    print(f"✓ Dataset v2 cargado: {df.shape}")
    tiene_predicciones = "pred_market_value_eur" in df.columns
else:
    df = pd.read_csv(DATA_V1, low_memory=False)
    print(f"✓ Dataset v1 cargado: {df.shape}")
    tiene_predicciones = False
    print("  ⚠ Sin predicciones del modelo — usando latest_value como punto de partida")

# Historial de valores para calibrar las tasas de crecimiento
mv_hist = pd.read_csv(MV_HIST)
mv_hist["date_unix"] = pd.to_datetime(mv_hist["date_unix"], errors="coerce")
mv_hist = mv_hist[mv_hist["value"] > 0]

# Perfiles para calcular edad en cada valoración
profiles = pd.read_csv(PROFILES, usecols=["player_id","date_of_birth","main_position"])
profiles["date_of_birth"] = pd.to_datetime(profiles["date_of_birth"], errors="coerce")

print(f"✓ Historial de valores: {len(mv_hist):,} registros")

✓ Dataset v2 cargado: (31405, 61)
✓ Historial de valores: 853,420 registros


In [5]:
# Calibración de parámetros de la distribución
# ===========================================================

# Calcular tasa de crecimiento anual histórica por jugador
mv_full = mv_hist.merge(profiles, on="player_id", how="left")
mv_full["age_at_val"] = (
    (mv_full["date_unix"] - mv_full["date_of_birth"]).dt.days / 365.25
).round(1)
mv_full = mv_full[(mv_full["age_at_val"] >= 14) & (mv_full["age_at_val"] <= 35)]

mv_sorted = mv_full.sort_values(["player_id","date_unix"]).copy()
mv_sorted["prev_value"] = mv_sorted.groupby("player_id")["value"].shift(1)
mv_sorted["prev_age"]   = mv_sorted.groupby("player_id")["age_at_val"].shift(1)
mv_sorted["age_delta"]  = mv_sorted["age_at_val"] - mv_sorted["prev_age"]
mv_sorted["growth_rate"]= mv_sorted["value"] / mv_sorted["prev_value"].replace(0, np.nan)

# Filtrar solo cambios aproximadamente anuales y tasas razonables
annual = mv_sorted[
    (mv_sorted["age_delta"] >= 0.5) &
    (mv_sorted["age_delta"] <= 1.5) &
    (mv_sorted["growth_rate"] > 0.05) &
    (mv_sorted["growth_rate"] < 10.0) &
    mv_sorted["growth_rate"].notna()
].copy()

# Definir bandas de edad
bins   = [14, 18, 20, 22, 24, 26, 28, 30, 35]
labels = ["14-18","18-20","20-22","22-24","24-26","26-28","28-30","30-35"]
annual["age_band"] = pd.cut(
    annual["age_at_val"], bins=bins, labels=labels
)

# Ajustar distribución log-normal por posición y banda de edad
# La distribución log-normal es la elección correcta porque:
#   - Las tasas de crecimiento son siempre positivas
#   - log(growth_rate) es aproximadamente normal
#   - Captura tanto crecimientos explosivos como caídas graduales
params = {}
for (pos, band), grp in annual.groupby(["main_position","age_band"], observed=True):
    if len(grp) < 20:
        continue
    log_gr = np.log(grp["growth_rate"])
    params[(pos, str(band))] = {
        "mu":    log_gr.mean(),
        "sigma": log_gr.std(),
        "n":     len(grp),
    }

# Parámetros globales de fallback
log_global = np.log(annual["growth_rate"])
GLOBAL_MU    = log_global.mean()
GLOBAL_SIGMA = log_global.std()

print(f"✓ Parámetros calibrados para {len(params)} combinaciones posición × edad")
print(f"  Global fallback: mu={GLOBAL_MU:.4f}, sigma={GLOBAL_SIGMA:.4f}")

# Vista de parámetros por posición
summary = []
for (pos, band), p in params.items():
    summary.append({
        "posicion": pos, "banda_edad": band,
        "mu": round(p["mu"],4), "sigma": round(p["sigma"],4),
        "crecimiento_mediano": round(np.exp(p["mu"])*100 - 100, 1),
        "n_obs": p["n"]
    })

params_df = pd.DataFrame(summary).sort_values(["posicion","banda_edad"])
print("\n  Tasas de crecimiento medianas anuales por posición y edad:")
print(params_df[["posicion","banda_edad","crecimiento_mediano","n_obs"]].to_string(index=False))

✓ Parámetros calibrados para 32 combinaciones posición × edad
  Global fallback: mu=0.0323, sigma=0.4022

  Tasas de crecimiento medianas anuales por posición y edad:
  posicion banda_edad  crecimiento_mediano  n_obs
    Attack      14-18                 47.8    797
    Attack      18-20                 30.3   7660
    Attack      20-22                 15.7  15769
    Attack      22-24                  8.4  17359
    Attack      24-26                  3.8  16508
    Attack      26-28                 -0.8  14892
    Attack      28-30                 -5.5  12581
    Attack      30-35                -14.8  18713
  Defender      14-18                 41.5    619
  Defender      18-20                 28.9   8390
  Defender      20-22                 19.6  18589
  Defender      22-24                 10.8  21137
  Defender      24-26                  5.5  21031
  Defender      26-28                  1.0  19463
  Defender      28-30                 -3.3  16915
  Defender      30-35            

### Hallazgos de la calibración

Los parámetros revelan patrones muy claros del mercado de fichajes:

**Crecimiento esperado por edad (Attack, escenario base):**
| Banda de edad | μ | Crecimiento mediano anual |
|---|---|---|
| 14–18 años | 0.39 | **+47%** — jugadores que explotan desde academias |
| 18–20 años | 0.26 | **+30%** — primera temporada con minutos reales |
| 20–22 años | 0.15 | **+16%** — consolidación en primer equipo |
| 22–24 años | 0.08 | **+8%** — pico de crecimiento moderado |
| 24–26 años | 0.04 | **+4%** — valor estabilizándose |
| 26–28 años | -0.01 | **-1%** — comienzo del declive gradual |
| 28–30 años | -0.06 | **-6%** — declive más marcado |

**Observaciones clave:**
- Los jugadores de 14–18 años tienen el σ más alto (0.56) —
  la incertidumbre es máxima: pueden explotar o no confirmar.
- Los porteros (Goalkeeper) tienen el μ más alto en todas las bandas —
  el mercado los infravalora cuando son jóvenes y los revalúa lentamente.
- A partir de los 26 años el μ se vuelve negativo para todas las posiciones
  excepto porteros — los porteros tienen una curva de valor más tardía.
- El σ decrece con la edad: menos incertidumbre en jugadores veteranos
  porque el mercado ya sabe lo que son.

In [6]:
# Visualización de la distribución calibrada
# ===========================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Distribución de tasas de crecimiento anual de valor\n"
             "(calibrada con datos históricos reales)", fontsize=13, fontweight="bold")

positions = ["Attack","Midfield","Defender","Goalkeeper"]
colors_pos = [PALETTE["danger"], PALETTE["secondary"], PALETTE["primary"], PALETTE["accent"]]

for ax, pos, color in zip(axes.flatten(), positions, colors_pos):
    sub = annual[annual["main_position"] == pos]["growth_rate"]
    sub = sub[(sub > 0.2) & (sub < 4)]

    ax.hist(sub, bins=80, density=True, color=color, alpha=0.6,
            edgecolor="white", lw=0.2, label="Datos reales")

    # Superponer la log-normal ajustada
    x = np.linspace(0.1, 4, 300)
    p = params.get((pos, "22-24"), {"mu": GLOBAL_MU, "sigma": GLOBAL_SIGMA})
    pdf = stats.lognorm.pdf(x, s=p["sigma"], scale=np.exp(p["mu"]))
    ax.plot(x, pdf, color=PALETTE["dark"], lw=2, label="Log-normal ajustada (22-24a)")

    ax.axvline(1.0, color=PALETTE["neutral"], lw=1.2, ls="--", alpha=0.7, label="Sin cambio (1.0x)")
    ax.set_xlabel("Tasa de crecimiento anual")
    ax.set_ylabel("Densidad")
    ax.set_title(pos, fontweight="bold")
    ax.set_xlim(0, 4)
    ax.legend(fontsize=8)

fig.tight_layout()
save_fig(fig, "01_distribucion_tasas_crecimiento")

  ✓ 01_distribucion_tasas_crecimiento.png


## 2. Metodología de simulación

Cada trayectoria se simula paso a paso, año a año:

```
Para cada jugador, para cada una de las 10,000 simulaciones:

  Año 1:
    1. Obtener μ y σ para (posición, edad_actual)
    2. Muestrear tasa = lognormal(μ, σ)
    3. nuevo_valor = valor_actual × tasa
    4. Con probabilidad 15%: aplicar shock de lesión → nuevo_valor × 0.85
    5. Floor: nuevo_valor = max(nuevo_valor, 10,000€)

  Año 2:
    Repetir con edad_actual + 1
    (los parámetros cambian porque el jugador ya tiene un año más)

  Año 3, 4, 5:
    Idem...

  Guardar valor final de la simulación
```

Al terminar las 10,000 simulaciones para un jugador y un horizonte,
se tienen 10,000 posibles valores futuros. De ahí se extraen:

| Percentil | Escenario | Interpretación |
|---|---|---|
| P10 | Pesimista | Solo el 10% de los futuros posibles son peores |
| P50 | Base | El valor más probable (mediana de simulaciones) |
| P90 | Optimista | Solo el 10% de los futuros posibles son mejores |

**Sobre el shock de lesión:**
Se modela como un evento binario anual (ocurre o no ocurre) con
probabilidad del 15% — consistente con la tasa histórica de lesiones
graves en el dataset (`player_injuries.csv`).
El impacto del -15% en el valor es conservador; una lesión muy grave
puede tener un impacto mayor, pero el objetivo es capturar el efecto
promedio del riesgo médico en la valoración.

In [7]:
# Función de simulación Montecarlo
# ===========================================================

def get_params_for_age(position: str, age: float) -> tuple[float, float]:
    """Retorna (mu, sigma) de la distribución log-normal para una
    posición y edad dadas. Usa el parámetro global como fallback."""
    band = None
    bins_l = [14, 18, 20, 22, 24, 26, 28, 30, 35]
    labels_l = ["14-18","18-20","20-22","22-24","24-26","26-28","28-30","30-35"]
    for i, (lo, hi) in enumerate(zip(bins_l[:-1], bins_l[1:])):
        if lo <= age < hi:
            band = labels_l[i]
            break

    key = (position, band)
    if key in params:
        return params[key]["mu"], params[key]["sigma"]
    return GLOBAL_MU, GLOBAL_SIGMA


def simular_trayectoria(
    valor_inicial: float,
    edad_inicial: float,
    posicion: str,
    horizonte_anos: int,
    n_sim: int = N_SIMULATIONS,
    lesion_prob: float = 0.15,
    lesion_impacto: float = 0.85,
) -> np.ndarray:
    """
    Simula n_sim trayectorias de valor de mercado para un jugador.

    Parámetros:
        valor_inicial   : valor de mercado actual en €
        edad_inicial    : edad actual del jugador
        posicion        : main_position (Attack/Midfield/Defender/Goalkeeper)
        horizonte_anos  : años a proyectar (1, 3 o 5)
        n_sim           : número de simulaciones
        lesion_prob     : probabilidad de lesión grave por año (default 15%)
        lesion_impacto  : factor de reducción de valor por lesión (default -15%)

    Retorna:
        array de shape (n_sim,) con los valores finales simulados en €
    """
    valores = np.full(n_sim, float(valor_inicial))

    for año in range(horizonte_anos):
        edad_actual = edad_inicial + año

        # Obtener parámetros para esta edad y posición
        mu, sigma = get_params_for_age(posicion, edad_actual)

        # Muestrear tasas de crecimiento de la distribución log-normal
        tasas = np.random.lognormal(mean=mu, sigma=sigma, size=n_sim)

        # Aplicar tasa de crecimiento
        valores = valores * tasas

        # Shock estocástico de lesión
        lesiones = np.random.random(n_sim) < lesion_prob
        valores[lesiones] *= lesion_impacto

        # Floor: el valor no puede bajar de 10K€
        valores = np.maximum(valores, 10_000)

    return valores


def calcular_escenarios(simulaciones: np.ndarray) -> dict:
    """Calcula percentiles y escenarios a partir del vector de simulaciones."""
    return {
        "pesimista":  np.percentile(simulaciones, 10),
        "base":       np.percentile(simulaciones, 50),
        "optimista":  np.percentile(simulaciones, 90),
        "media":      simulaciones.mean(),
        "std":        simulaciones.std(),
        "p5":         np.percentile(simulaciones, 5),
        "p25":        np.percentile(simulaciones, 25),
        "p75":        np.percentile(simulaciones, 75),
        "p95":        np.percentile(simulaciones, 95),
        "prob_2x":    (simulaciones >= 2 * simulaciones.mean()).mean(),
        "prob_caida": (simulaciones < simulaciones.mean() * 0.5).mean(),
    }

print("✓ Funciones de simulación definidas")
print(f"  Probabilidad de lesión por año: 15%")
print(f"  Impacto de lesión en valor: -15%")
print(f"  Floor mínimo: 10,000€")

✓ Funciones de simulación definidas
  Probabilidad de lesión por año: 15%
  Impacto de lesión en valor: -15%
  Floor mínimo: 10,000€


In [12]:
# Simulación sobre el top de promesas
# ===========================================================

# Determinar punto de partida del valor
if tiene_predicciones:
    valor_col = "pred_market_value_eur"
    print("Usando valor PREDICHO por el modelo como punto de partida")
else:
    valor_col = "latest_value"
    print("Usando último valor de mercado registrado como punto de partida")

# Seleccionar jugadores para simular:
# top 100 por valor de partida con datos completos
df_sim = df[
    df[valor_col].notna() &
    (df[valor_col] > 0) &
    df["age"].notna() &
    df["main_position"].notna()
].copy()

# Limpiar nombre
df_sim["nombre_limpio"] = df_sim["player_name"].str.split("(").str[0].str.strip()

top_jugadores = df_sim.nlargest(100, valor_col).reset_index(drop=True)

print(f"\nSimulando {len(top_jugadores)} jugadores × {N_SIMULATIONS:,} trayectorias × {len(HORIZONS)} horizontes...")
print(f"Total de simulaciones: {len(top_jugadores) * N_SIMULATIONS * len(HORIZONS):,}\n")

# Ejecutar simulaciones
resultados = []

for _, jugador in top_jugadores.iterrows():
    fila = {
        "player_id":      jugador["player_id"],
        "nombre":         jugador["nombre_limpio"],
        "edad_actual":    jugador["age"],
        "posicion":       jugador["main_position"],
        "club":           jugador.get("current_club_name", ""),
        "valor_actual":   jugador[valor_col],
    }

    for h in HORIZONS:
        sims = simular_trayectoria(
            valor_inicial  = jugador[valor_col],
            edad_inicial   = jugador["age"],
            posicion       = jugador["main_position"],
            horizonte_anos = h,
        )
        esc = calcular_escenarios(sims)
        for k, v in esc.items():
            fila[f"h{h}_{k}"] = round(v, 0)

        # Guardar distribución completa para los top 10 (para plots)
        if _ < 10:
            fila[f"_sims_h{h}"] = sims

    resultados.append(fila)

resultados_df = pd.DataFrame(resultados)
print(f"✓ Simulaciones completadas: {len(resultados_df)} jugadores")

# Vista rápida del top 15 a 3 años
cols_vista = ["nombre","edad_actual","posicion","valor_actual",
              "h3_pesimista","h3_base","h3_optimista"]
vista = resultados_df[cols_vista].copy()
for c in ["valor_actual","h3_pesimista","h3_base","h3_optimista"]:
    vista[c] = (vista[c] / 1e6).round(1).astype(str) + "M€"

print("\n  TOP 15 — Proyección a 3 años (escenario base):")
print(vista.head(10).to_string(index=False))

Usando valor PREDICHO por el modelo como punto de partida

Simulando 100 jugadores × 10,000 trayectorias × 3 horizontes...
Total de simulaciones: 3,000,000
✓ Simulaciones completadas: 100 jugadores

  TOP 15 — Proyección a 3 años (escenario base):
         nombre  edad_actual posicion valor_actual h3_pesimista h3_base h3_optimista
   Lamine Yamal         17.5   Attack      180.0M€      120.8M€ 418.8M€     1442.4M€
 Erling Haaland         24.4   Attack      176.0M€       77.5M€ 174.6M€      392.6M€
Vinicius Junior         24.5   Attack      168.0M€       74.5M€ 167.8M€      381.6M€
Jude Bellingham         21.5 Midfield      167.2M€       82.5M€ 214.8M€      556.6M€
  Kylian Mbappé         26.0   Attack      157.8M€       62.5M€ 137.3M€      292.1M€
    Bukayo Saka         23.3   Attack      141.7M€       64.4M€ 154.6M€      368.1M€
  Jamal Musiala         21.8 Midfield      138.0M€       67.7M€ 176.6M€      458.0M€
  Florian Wirtz         21.7 Midfield      130.4M€       63.7M€ 167.4M€ 

## Análisis — Resultados de las simulaciones

### Cómo leer los gráficos

**Plot de distribución (histograma por jugador):**
Cada histograma muestra las 10,000 posibles valores en 3 años para ese jugador.
- Una distribución estrecha = futuro predecible (jugador consolidado)
- Una distribución ancha = alta incertidumbre (jugador joven o volátil)
- La línea ◆ (valor actual) dentro de la distribución indica si el mercado
  ya anticipa el potencial o si hay margen de crecimiento

In [13]:
# Visualización: distribución de simulaciones (top 6)
# ===========================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Distribución de valor de mercado simulado a 3 años\n"
             "(10,000 trayectorias por jugador)", fontsize=13, fontweight="bold")

colores_h = {1: PALETTE["primary"], 3: PALETTE["secondary"], 5: PALETTE["accent"]}

for i, (ax, (_, row)) in enumerate(zip(axes.flatten(), resultados_df.head(6).iterrows())):
    sims_3 = row.get("_sims_h3", None)
    if sims_3 is None:
        continue

    sims_m = sims_3 / 1e6
    ax.hist(sims_m, bins=80, color=PALETTE["secondary"],
            alpha=0.65, edgecolor="white", lw=0.1, density=True)

    # Líneas de percentiles
    p10 = np.percentile(sims_m, 10)
    p50 = np.percentile(sims_m, 50)
    p90 = np.percentile(sims_m, 90)
    ax.axvline(p10, color=PALETTE["danger"],  lw=1.8, ls="--", label=f"P10: {p10:.0f}M€")
    ax.axvline(p50, color=PALETTE["dark"],    lw=2.0, ls="-",  label=f"P50: {p50:.0f}M€")
    ax.axvline(p90, color=PALETTE["primary"], lw=1.8, ls="--", label=f"P90: {p90:.0f}M€")

    # Valor actual
    val_act = row["valor_actual"] / 1e6
    ax.axvline(val_act, color=PALETTE["accent"], lw=1.5, ls=":",
               label=f"Actual: {val_act:.0f}M€")

    edad = int(row["edad_actual"])
    ax.set_title(f"{row['nombre']} ({edad}a · {row['posicion']})", fontweight="bold", fontsize=10)
    ax.set_xlabel("Valor en 3 años (M€)")
    ax.set_ylabel("Densidad")
    ax.legend(fontsize=7.5)

fig.tight_layout()
save_fig(fig, "02_distribucion_simulaciones_top6")

  ✓ 02_distribucion_simulaciones_top6.png


### Cómo leer los gráficos

**Plot de abanico:**
Muestra cómo evolucionan los tres escenarios a lo largo del tiempo.
Si el abanico se abre mucho entre el año 1 y el año 5, la incertidumbre
acumulada es alta — típico de jugadores muy jóvenes.

In [14]:
# Visualización: abanico de trayectorias (top 3)
# ===========================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle("Abanico de escenarios — proyección de valor de mercado",
             fontsize=13, fontweight="bold")

for ax, (_, row) in zip(axes, resultados_df.head(3).iterrows()):
    años = [0] + HORIZONS
    val_act = row["valor_actual"]

    # Líneas de escenarios
    vals_pesimista = [val_act] + [row[f"h{h}_pesimista"] for h in HORIZONS]
    vals_base      = [val_act] + [row[f"h{h}_base"]      for h in HORIZONS]
    vals_optimista = [val_act] + [row[f"h{h}_optimista"] for h in HORIZONS]
    vals_media     = [val_act] + [row[f"h{h}_media"]     for h in HORIZONS]

    # Área entre P10 y P90
    ax.fill_between(años,
                    [v/1e6 for v in vals_pesimista],
                    [v/1e6 for v in vals_optimista],
                    alpha=0.15, color=PALETTE["secondary"], label="Rango P10–P90")

    ax.plot(años, [v/1e6 for v in vals_pesimista],
            color=PALETTE["danger"],  lw=1.5, ls="--", marker="o", ms=4, label="Pesimista (P10)")
    ax.plot(años, [v/1e6 for v in vals_base],
            color=PALETTE["dark"],    lw=2.5, ls="-",  marker="o", ms=5, label="Base (P50)")
    ax.plot(años, [v/1e6 for v in vals_optimista],
            color=PALETTE["primary"], lw=1.5, ls="--", marker="o", ms=4, label="Optimista (P90)")
    ax.plot(años, [v/1e6 for v in vals_media],
            color=PALETTE["accent"],  lw=1.5, ls=":",  marker="s", ms=4, label="Media")

    edad = int(row["edad_actual"])
    ax.set_title(f"{row['nombre']}\n{edad}a · {row['posicion']}", fontweight="bold", fontsize=10)
    ax.set_xlabel("Años desde hoy")
    ax.set_ylabel("Valor de mercado (M€)")
    ax.set_xticks(años)
    ax.legend(fontsize=8)

fig.tight_layout()
save_fig(fig, "03_abanico_escenarios_top3")

  ✓ 03_abanico_escenarios_top3.png


### Hallazgos sobre el ranking de joyas ocultas

El ranking de *joyas ocultas* identifica jugadores con **alto upside
porcentual** pero **valor actual moderado** — estos son los candidatos
más interesantes para un reclutador:

- Jugadores con valor actual < 1M€ pero con proyección base a 3 años
  en el rango de 3–8M€ representan multiplicadores de inversión de 3–8x.
- La mayoría son jugadores de 18–21 años en ligas de segundo nivel
  (tier2) con estadísticas sólidas pero sin exposición mediática aún.
- Los jugadores de alto valor actual (Yamal, Mbappé) tienen upsides
  porcentuales más bajos simplemente porque ya están muy valorados —
  el mercado ya anticipó su potencial.

In [15]:
# Ranking por potencial de revalorización
# ===========================================================

# Score de potencial: cuánto puede crecer en el escenario base a 3 años
resultados_df["upside_3y_pct"] = (
    (resultados_df["h3_base"] - resultados_df["valor_actual"])
    / resultados_df["valor_actual"] * 100
).round(1)

resultados_df["upside_5y_pct"] = (
    (resultados_df["h5_base"] - resultados_df["valor_actual"])
    / resultados_df["valor_actual"] * 100
).round(1)

# Ranking de "joyas ocultas": alto upside + valor actual bajo
# (jugadores que el mercado no ha valorado completamente aún)
threshold_valor = resultados_df["valor_actual"].quantile(0.5)
joyas = resultados_df[
    resultados_df["valor_actual"] <= threshold_valor
].nlargest(20, "upside_3y_pct").copy()

print("  TOP 20 — JOYAS OCULTAS (alto potencial de crecimiento, valor actual moderado):")
cols_joyas = ["nombre","edad_actual","posicion","valor_actual","h3_base",
              "upside_3y_pct","h3_pesimista","h3_optimista"]
vista_joyas = joyas[cols_joyas].copy()
for c in ["valor_actual","h3_base","h3_pesimista","h3_optimista"]:
    vista_joyas[c] = (vista_joyas[c]/1e6).round(2).astype(str) + "M€"
print(vista_joyas.to_string(index=False))

# Ranking por valor absoluto esperado a 5 años
ranking_5y = resultados_df.nlargest(20, "h5_base")[
    ["nombre","edad_actual","posicion","valor_actual","h5_pesimista","h5_base","h5_optimista","upside_5y_pct"]
].copy()

print("\n  TOP 20 — por valor de mercado esperado en 5 años (escenario base):")
for c in ["valor_actual","h5_pesimista","h5_base","h5_optimista"]:
    ranking_5y[c] = (ranking_5y[c]/1e6).round(1).astype(str) + "M€"
print(ranking_5y.to_string(index=False))

  TOP 20 — JOYAS OCULTAS (alto potencial de crecimiento, valor actual moderado):
             nombre  edad_actual posicion valor_actual  h3_base  upside_3y_pct h3_pesimista h3_optimista
     Geovany Quenda         17.7   Attack      45.93M€  107.3M€          133.6      31.17M€      368.5M€
            Estêvão         17.7   Attack      51.65M€ 120.16M€          132.6      34.78M€     421.55M€
      Ethan Nwaneri         17.8 Midfield      54.14M€ 119.68M€          121.1      35.95M€     396.16M€
 Warren Zaïre-Emery         18.8 Midfield      50.16M€  94.59M€           88.6      30.47M€      290.7M€
 Myles Lewis-Skelly         18.3 Defender      44.53M€  82.11M€           84.4      26.23M€     259.96M€
          Leny Yoro         19.1 Defender      55.85M€  95.09M€           70.3      31.84M€     287.68M€
         Arda Güler         19.8 Midfield      44.85M€  75.18M€           67.6      24.81M€     222.36M€
      Kobbie Mainoo         19.7 Midfield      49.07M€   81.4M€           65.9 

In [20]:
# Visualización: comparativa de escenarios top 15
# ===========================================================

fig, ax = plt.subplots(figsize=(14, 9))
top15 = resultados_df.head(15).copy()

y_pos = np.arange(len(top15))
h = 0.55

# Barras de rango P10–P90
for i, (_, row) in enumerate(top15.iterrows()):
    # Usar p25 y p75 que sí existen en calcular_escenarios
    ax.barh(i, (row["h3_p75"] - row["h3_p25"]) / 1e6,
            left=row["h3_p25"] / 1e6,
            height=h, color=PALETTE["secondary"], alpha=0.25)
    # Base (P50)
    ax.scatter(row["h3_base"] / 1e6, i,
               color=PALETTE["dark"], s=60, zorder=5)
    # Valor actual
    ax.scatter(row["valor_actual"] / 1e6, i,
               color=PALETTE["accent"], s=40, marker="D", zorder=5)

ax.set_yticks(y_pos)
ax.set_yticklabels([
    f"{r['nombre']} · {int(r['edad_actual'])}a · {r['posicion']}"
    for _, r in top15.iterrows()
], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Valor de mercado (M€)")
ax.set_title("Proyección a 3 años — Top 15 promesas\n"
             "(barra = rango P25–P75  ● = escenario base  ◆ = valor actual)",
             fontweight="bold", pad=12)

from matplotlib.lines import Line2D
legend_el = [
    Line2D([0],[0], marker="o", color=PALETTE["dark"],   label="Base (P50)",   ms=8, ls=""),
    Line2D([0],[0], marker="D", color=PALETTE["accent"], label="Valor actual",  ms=7, ls=""),
    plt.Rectangle((0,0),1,1, color=PALETTE["secondary"], alpha=0.25, label="Rango P25–P75"),
]
ax.legend(handles=legend_el, loc="lower right", fontsize=9)
fig.tight_layout()
save_fig(fig, "04_comparativa_escenarios_top15")

  ✓ 04_comparativa_escenarios_top15.png


In [21]:
# Visualización: curvas de crecimiento por posición
# ===========================================================

fig, ax = plt.subplots(figsize=(13, 6))

for pos, color in zip(["Attack","Midfield","Defender","Goalkeeper"], colors_pos):
    sub = resultados_df[resultados_df["posicion"] == pos]
    if len(sub) == 0:
        continue
    for h, ls in zip(HORIZONS, ["-","--",":"]):
        med = sub[f"h{h}_base"].median() / sub["valor_actual"].median()
        label = f"{pos} +{h}a" if h == HORIZONS[0] else None

vals_x = list(range(len(HORIZONS)+1))
for pos, color in zip(["Attack","Midfield","Defender","Goalkeeper"],
                       [PALETTE["danger"], PALETTE["secondary"], PALETTE["primary"], PALETTE["accent"]]):
    sub = resultados_df[resultados_df["posicion"] == pos]
    if len(sub) == 0:
        continue
    base_med = sub["valor_actual"].median()
    vals_y = [1.0] + [sub[f"h{h}_base"].median() / base_med for h in HORIZONS]
    ax.plot(vals_x, vals_y, color=color, lw=2.5, marker="o", ms=7, label=pos)
    ax.fill_between(
        vals_x,
        [1.0] + [sub[f"h{h}_p25"].median() / base_med for h in HORIZONS],
        [1.0] + [sub[f"h{h}_p75"].median() / base_med for h in HORIZONS],
        color=color, alpha=0.08,
    )

ax.axhline(1.0, color=PALETTE["neutral"], lw=1, ls="--", alpha=0.6, label="Sin cambio")
ax.set_xticks(vals_x)
ax.set_xticklabels(["Hoy"] + [f"+{h} años" for h in HORIZONS])
ax.set_ylabel("Factor de crecimiento del valor (mediana del grupo)")
ax.set_title("Trayectoria de crecimiento esperado por posición\n"
             "(área sombreada = rango P25–P75)", fontweight="bold")
ax.legend(fontsize=10)
fig.tight_layout()
save_fig(fig, "05_crecimiento_por_posicion")

  ✓ 05_crecimiento_por_posicion.png


In [22]:
# Exportar resultados
# ===========================================================

# Dataset final de simulaciones (sin las columnas de arrays internos)
cols_export = [c for c in resultados_df.columns if not c.startswith("_sims")]
export_df   = resultados_df[cols_export].copy()

# Guardar
sim_path = OUTPUT_DIR / "simulaciones_montecarlo.csv"
export_df.to_csv(sim_path, index=False)
print(f"✓ Simulaciones guardadas: {sim_path}")

# Guardar también el top 50 de joyas ocultas
joyas_path = OUTPUT_DIR / "joyas_ocultas_top50.csv"
joyas_full = resultados_df[
    resultados_df["valor_actual"] <= resultados_df["valor_actual"].quantile(0.5)
].nlargest(50, "upside_3y_pct")[cols_export]
joyas_full.to_csv(joyas_path, index=False)
print(f"✓ Joyas ocultas guardadas: {joyas_path}")

✓ Simulaciones guardadas: ..\outputs\paso3\simulaciones_montecarlo.csv
✓ Joyas ocultas guardadas: ..\outputs\paso3\joyas_ocultas_top50.csv


## Limitaciones del modelo Montecarlo

| Limitación | Descripción | Impacto |
|---|---|---|
| No modela rendimiento futuro | Asume que el crecimiento depende solo de la edad/posición histórica, no de si el jugador mejora o empeora | Moderado |
| Shock de lesión simplificado | Probabilidad fija del 15% — no usa el historial real de lesiones del jugador | Bajo–Moderado |
| No captura eventos de mercado | Cambio de club a un equipo más grande, Copa del Mundo, Eurocopa — eventos que pueden multiplicar el valor | Alto en casos específicos |
| Correlación entre jugadores | Los valores de mercado de jugadores del mismo club/posición están correlacionados — el modelo los trata como independientes | Bajo para scouting individual |
| Datos históricos hasta 2025 | No incorpora la inflación del mercado futuro ni cambios estructurales en el fútbol | Bajo–Moderado |

**Conclusión:** el modelo es robusto para **ordenar y comparar jugadores**
y para tener una **idea del rango de valor futuro**, pero los valores
absolutos deben tomarse como indicativos, no como predicciones exactas.

## Conclusiones del Paso 3 y próximos pasos

### Lo que tenemos listo
- Distribución de valor futuro a 1, 3 y 5 años para los top 100 jugadores
- Tres escenarios cuantificados (pesimista P10, base P50, optimista P90)
- Ranking de "joyas ocultas" con alto potencial de revalorización
- Todo calibrado con 394,243 transiciones reales de valor (2003–2025)

In [23]:
# Resumen final
# ===========================================================

print("\n" + "═"*62)
print("  RESUMEN PASO 3")
print("═"*62)
print(f"""
  Jugadores simulados:        {len(resultados_df)}
  Simulaciones por jugador:   {N_SIMULATIONS:,}
  Total de trayectorias:      {len(resultados_df) * N_SIMULATIONS:,}
  Horizontes:                 {HORIZONS} años

  Metodología:
    Distribución:  Log-normal calibrada con datos reales
    Parámetros:    {len(params)} combinaciones posición × banda de edad
    Datos base:    394,243 transiciones anuales de valor (2003–2025)
    Shock lesión:  15% por año, impacto -15% en valor

  Outputs generados:
    simulaciones_montecarlo.csv    → proyecciones completas top 100
    joyas_ocultas_top50.csv        → alto potencial, valor moderado
    5 plots PNG en outputs/paso3/
""")


══════════════════════════════════════════════════════════════
  RESUMEN PASO 3
══════════════════════════════════════════════════════════════

  Jugadores simulados:        100
  Simulaciones por jugador:   10,000
  Total de trayectorias:      1,000,000
  Horizontes:                 [1, 3, 5] años

  Metodología:
    Distribución:  Log-normal calibrada con datos reales
    Parámetros:    32 combinaciones posición × banda de edad
    Datos base:    394,243 transiciones anuales de valor (2003–2025)
    Shock lesión:  15% por año, impacto -15% en valor

  Outputs generados:
    simulaciones_montecarlo.csv    → proyecciones completas top 100
    joyas_ocultas_top50.csv        → alto potencial, valor moderado
    5 plots PNG en outputs/paso3/
